<a href="https://colab.research.google.com/github/nagayada/Nag_New_Repo/blob/main/Mini_Assignment_1_Y_Nagaraju.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install PySpark
!pip install pyspark

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,when,datediff,avg,sum, count
from pyspark.sql.types import IntegerType,DoubleType,DateType

spark = SparkSession.builder.appName("LungCancerAnalysis").getOrCreate()
print("SparkSession created successfully")

SparkSession created successfully


In [29]:
df = spark.read.csv('/content/Lung Cancer.csv',header=True,inferSchema=True)
df.show(5)
df.printSchema()

+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
| id| age|gender|    country|diagnosis_date|cancer_stage|family_history|smoking_status| bmi|cholesterol_level|hypertension|asthma|cirrhosis|other_cancer|treatment_type|end_treatment_date|survived|
+---+----+------+-----------+--------------+------------+--------------+--------------+----+-----------------+------------+------+---------+------------+--------------+------------------+--------+
|  1|64.0|  Male|     Sweden|    2016-04-05|     Stage I|           Yes|Passive Smoker|29.4|              199|           0|     0|        1|           0|  Chemotherapy|        2017-09-10|       0|
|  2|50.0|Female|Netherlands|    2023-04-20|   Stage III|           Yes|Passive Smoker|41.2|              280|           1|     1|        0|           0|       Surgery|        2024-06-17|       1|
|  3|65.0|Femal

In [45]:
def clean_lung_cancer_data(df):
    # 1. Remove duplicate rows
    df_cleaned = df.drop_duplicates()
    print(f"Number of rows after removing duplicates: {df_cleaned.count()}")

    # 2. Ensure correct data types for date columns
    # 'diagnosis_date' and 'end_treatment_date' are already of DateType from inferSchema=True.
    # Explicitly re-casting ensures consistency.
    date_cols_to_cast = ['diagnosis_date', 'end_treatment_date']
    for col_name in date_cols_to_cast:
        if col_name in df_cleaned.columns:
            df_cleaned = df_cleaned.withColumn(col_name, col(col_name).cast(DateType()))
    print("Date columns explicitly cast to DateType.")

    if 'gender' in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(
            'gender',
            when(col('gender').cast('string').ilike('Male'), 1)
            .when(col('gender').cast('string').ilike('Female'), 0)
            .otherwise(None) # Handle unexpected values, can be null
        ).withColumn('gender', col('gender').cast(IntegerType()))
    print("Gender column converted to 1/0.")

    # Convert 'family_history'
    if 'family_history' in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(
            'family_history',
            when(col('family_history').cast('string').ilike('Yes'), 1)
            .when(col('family_history').cast('string').ilike('No'), 0)
            .otherwise(None) # Handle unexpected values, can be null
        ).withColumn('family_history', col('family_history').cast(IntegerType()))
    print("Family_history column converted to 1/0.")

    print("\nSchema after cleaning and type conversions:")
    df_cleaned.printSchema()
    return df_cleaned

In [46]:
df_cleaned = clean_lung_cancer_data(df)

# Display cleaned data info and first rows
print("\nFirst 5 rows of the cleaned DataFrame:")
df_cleaned.show(5)

Number of rows after removing duplicates: 890000
Date columns explicitly cast to DateType.
Gender column converted to 1/0.
Family_history column converted to 1/0.

Schema after cleaning and type conversions:
root
 |-- id: integer (nullable = true)
 |-- age: double (nullable = true)
 |-- gender: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- diagnosis_date: date (nullable = true)
 |-- cancer_stage: string (nullable = true)
 |-- family_history: integer (nullable = true)
 |-- smoking_status: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- cholesterol_level: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- asthma: integer (nullable = true)
 |-- cirrhosis: integer (nullable = true)
 |-- other_cancer: integer (nullable = true)
 |-- treatment_type: string (nullable = true)
 |-- end_treatment_date: date (nullable = true)
 |-- survived: integer (nullable = true)


First 5 rows of the cleaned DataFrame:
+---+----+------+---------

# Task 2: Calculate Treatment Duration and Average Duration per Treatment Type

In [47]:
def calculate_treatment_duration(df_cleaned):
    df_with_duration = df_cleaned.withColumn(
        'treatment_duration_days',
        datediff(col('end_treatment_date'), col('diagnosis_date'))
    )

    df_with_duration = df_with_duration.filter(col('treatment_duration_days').isNotNull() & (col('treatment_duration_days') >= 0))

    print("\nDataFrame with 'treatment_duration_days':")
    df_with_duration.select('diagnosis_date', 'end_treatment_date', 'treatment_duration_days', 'treatment_type').show(5)

    avg_duration_per_treatment = df_with_duration.groupBy('treatment_type').agg(
        avg('treatment_duration_days').alias('average_treatment_duration_days')
    )

    print("\nAverage treatment duration for each treatment type:")
    avg_duration_per_treatment.show()

    return avg_duration_per_treatment

In [48]:
avg_treatment_duration_df = calculate_treatment_duration(df_cleaned)


DataFrame with 'treatment_duration_days':
+--------------+------------------+-----------------------+--------------+
|diagnosis_date|end_treatment_date|treatment_duration_days|treatment_type|
+--------------+------------------+-----------------------+--------------+
|    2017-06-30|        2019-04-06|                    645|       Surgery|
|    2015-02-13|        2016-12-18|                    674|      Combined|
|    2016-07-29|        2017-08-27|                    394|  Chemotherapy|
|    2014-06-16|        2016-01-14|                    577|      Combined|
|    2017-12-25|        2019-03-31|                    461|  Chemotherapy|
+--------------+------------------+-----------------------+--------------+
only showing top 5 rows

Average treatment duration for each treatment type:
+--------------+-------------------------------+
|treatment_type|average_treatment_duration_days|
+--------------+-------------------------------+
|     Radiation|             458.40320462900917|
|  Chemot

Task 3: Smoking Status Group with Highest

In [49]:
def get_smoking_status_highest_survival(df_cleaned):
    survival_rates = df_cleaned.groupBy('Smoking_Status').agg(
        (sum(col('survived')) / count(col('id'))).alias('survival_rate')
    )

    # Order by survival_rate in descending order and get the top one
    highest_survival_group = survival_rates.orderBy(col('survival_rate').desc()).first()

    print("\nSmoking Status Group with the Highest Survival Rate:")
    if highest_survival_group:
        print(f"Smoking Status: {highest_survival_group['Smoking_Status']}")
        print(f"Survival Rate: {highest_survival_group['survival_rate']:.2%}")
    else:
        print("No smoking status groups found or data is insufficient.")

    return highest_survival_group

In [50]:
highest_survival_smoking_status = get_smoking_status_highest_survival(df_cleaned)


Smoking Status Group with the Highest Survival Rate:
Smoking Status: Never Smoked
Survival Rate: 22.09%


Task 4: Top Three Countries with Highest Percentage of Stage IV Diagnoses

In [51]:
from pyspark.sql.functions import col, count, lit, round

def get_top_countries_stage_iv(df_cleaned):
    total_patients_per_country = df_cleaned.groupBy('country').agg(
        count(col('id')).alias('total_patients')
    )

    # Filter for Stage IV patients and count them per country
    stage_iv_patients_per_country = df_cleaned.filter(col('cancer_stage') == 'Stage IV').groupBy('country').agg(
        count(col('id')).alias('stage_iv_patients')
    )

    # Join the two DataFrames and calculate the percentage
    country_stage_iv_percentage = total_patients_per_country.join(
        stage_iv_patients_per_country, on='country', how='left'
    ).fillna(0) # Fill NaN with 0 for countries with no Stage IV patients

    country_stage_iv_percentage = country_stage_iv_percentage.withColumn(
        'percentage_stage_iv',
        round((col('stage_iv_patients') / col('total_patients')) * 100, 2)
    )

    # Get the top three countries
    top_three_countries = country_stage_iv_percentage.orderBy(col('percentage_stage_iv').desc()).limit(3)

    print("\nTop three countries with the highest percentage of patients diagnosed in Stage IV:")
    top_three_countries.show()

    return top_three_countries


In [52]:
top_countries_stage_iv_df = get_top_countries_stage_iv(df_cleaned)


Top three countries with the highest percentage of patients diagnosed in Stage IV:
+--------------+--------------+-----------------+-------------------+
|       country|total_patients|stage_iv_patients|percentage_stage_iv|
+--------------+--------------+-----------------+-------------------+
|        Greece|         33052|             8429|               25.5|
|       Croatia|         33138|             8426|              25.43|
|Czech Republic|         32885|             8317|              25.29|
+--------------+--------------+-----------------+-------------------+



Task 5: Filtered Patients Analysis

In [54]:
from pyspark.sql.functions import col, avg

def analyze_filtered_patients(df_cleaned):
    filtered_patients = df_cleaned.filter(
        (col('gender') == 1) &  # Male (assuming 1 for Male after cleaning)
        ((col('cancer_stage') == 'Stage III') | (col('cancer_stage') == 'Stage IV')) & \
        (col('family_history') == 1) & # Assuming 'Yes' for family history
        (col('smoking_status') == 'Current Smoker') & \
        (col('bmi') > 30) & \
        (col('survived') == 1)  # Survived
    )

    # Count the number of filtered patients
    num_filtered_patients = filtered_patients.count()

    if num_filtered_patients == 0:
        print("No patients found matching all criteria.")
        return None, None

    print(f"\nNumber of patients matching all criteria: {num_filtered_patients}")

    # Calculate average age
    average_age = filtered_patients.agg(avg('age')).collect()[0][0]
    print(f"Average age of these patients: {average_age:.2f}")

    # Calculate percentage of these patients who had hypertension
    hypertension_count = filtered_patients.filter(col('hypertension') == 1).count()
    percentage_hypertension = (hypertension_count / num_filtered_patients) * 100
    print(f"Percentage of these patients who had hypertension: {percentage_hypertension:.2f}%")

    return average_age, percentage_hypertension


In [55]:
average_age_filtered, percentage_hypertension_filtered = analyze_filtered_patients(df_cleaned)



Number of patients matching all criteria: 3194
Average age of these patients: 55.18
Percentage of these patients who had hypertension: 74.77%
